# 🚀 Fine-Tune Llama-3 for AI Resume Tailoring (Free Colab GPU)

This notebook trains a high-precision **Resume Architect model** using **Unsloth (2x faster, 70% less VRAM)** on a free Google Colab T4 GPU.

### Workflow:
1. **Load Llama-3-8B-Instruct** in 4-bit QLoRA
2. **Train on resume customization dataset** (`training_data.jsonl`)
3. **Export directly to GGUF format** (`q4_k_m`)
4. **Download and import into your local Ollama** with `ollama create resume-tailor -f Modelfile`

### Step 1: Install Unsloth and Training Dependencies

In [ ]:
%%capture
# Install Unsloth for lightning-fast QLoRA training
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.28" "trl<0.9.0" peft accelerate bitsandbytes datasets

### Step 2: Load Llama-3 8B with 4-bit Quantization

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096 # Supports long job descriptions + candidate context
dtype = None # Auto detection (Float16 for T4, Bfloat16 for Ampere+)
load_in_4bit = True # 4bit quantization to drastically reduce memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Attach LoRA Adapters targeting all projection layers
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### Step 3: Upload Training Dataset (`training_data.jsonl`)
Run the cell below and select your `training_data.jsonl` file generated from the Resume Builder app.

In [ ]:
from google.colab import files
import os

print("Upload your training_data.jsonl file:")
uploaded = files.upload()
dataset_filename = list(uploaded.keys())[0]
print(f"Uploaded dataset: {dataset_filename}")

### Step 4: Format Dataset with Llama-3 Chat Template

In [ ]:
from datasets import load_dataset

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files={"train": dataset_filename})
dataset = dataset.map(formatting_prompts_func, batched = True)

### Step 5: Train with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # ~2-3 epochs depending on dataset size
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("Training Finished Successfully!")

### Step 6: Export Directly to GGUF for Ollama
Unsloth can export directly to quantized GGUF format with 1 line of code.

In [ ]:
# Export to Q4_K_M GGUF (optimal balance of quality and speed)
model.save_pretrained_gguf("resume_tailor_model", tokenizer, quantization_method = "q4_k_m")

print("Model converted to GGUF format!")

### Step 7: Download the GGUF Model to your Computer

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob("resume_tailor_model/*.gguf")
if gguf_files:
    target_file = gguf_files[0]
    print(f"Downloading {target_file}...")
    files.download(target_file)
else:
    print("Looking for root gguf file:")
    root_ggufs = glob.glob("*.gguf")
    if root_ggufs:
        files.download(root_ggufs[0])

### Step 8: Load into your Local Ollama
Once downloaded:
1. Move the `.gguf` file to your `backend/finetuning/` folder.
2. Run:
   ```bash
   python backend/finetuning/export_ollama.py --model-name resume-tailor
   ```
3. In the Resume Builder web app, select `resume-tailor` from the model dropdown!